# Experimento A, segunda versão: mínimo e máximo (apêndice)
A avaliação do notebook `imputacoes_mesma_prova_tratados` (os 3 métodos na mesma prova, a 5, 10, 20 e 30% de falhas, com o KNN padronizado) nas bases do mínimo e do máximo, tratadas como a média (notebook `outliers_min_max`).
- **Mesma prova também entre as estatísticas:** o sorteio fica nos meses completos das três bases tratadas (mínimo, máximo e média). A média roda de novo nesses meses, para as 3 estatísticas serem avaliadas nas mesmas células. Por isso, os números da média aqui são um pouco diferentes dos do `imputacoes_mesma_prova_tratados`, que sorteia nos meses completos só dela.
- **O mesmo KNN:** o k e os pesos escolhidos na média (lidos do Excel do `imputacoes_mesma_prova_tratados`). Uma seção confere se, no mínimo e no máximo, a escolha seria a mesma.

# Set Up

In [1]:
import pandas as pd
from pcj.utils import dados, BASE_DIR
from pcj.metricas_avaliacao import avaliacao_mesma_prova, metricas_juntando

In [2]:
# As 3 bases tratadas, num dicionário com a chave da estatística
# (fr"...": texto cru, por causa das barras do caminho, e com {estatistica} trocado pelo valor)
estatisticas = ["min", "med", "max"]
bases = {}
for estatistica in estatisticas:
    bases[estatistica], _, variaveis = dados(fr"data\processed\dados_{estatistica}_tratados.xlsx")

# Resultados da média, gravados pelo notebook imputacoes_mesma_prova_tratados (a escolha do k)
caminho_tratada = BASE_DIR / "data" / "output" / "imputacoes_mesma_prova_tratados.xlsx"

# Taxas de falhas e métodos (os padrões da avaliacao_mesma_prova)
taxas = [0.05, 0.1, 0.2, 0.3]
metodos = ["media", "mediana", "knn"]

# Mesma prova
Os meses completos (sem NaN) de cada base e os comuns às três. Os meses completos que não são comuns continuam na base que o imputador vê (e na padronização do KNN), mas não são sorteados (`meses_sorteio`).

In [3]:
# Datas dos meses completos de cada base
completos = {}
for estatistica in estatisticas:
    completos[estatistica] = bases[estatistica].dropna(subset=variaveis)["Data"]
    print(f"Meses completos, {estatistica}:", len(completos[estatistica]))

# Datas completas nas três: isin marca True nas datas que também estão na outra lista
na_min = completos["med"].isin(completos["min"])
na_max = completos["med"].isin(completos["max"])
datas_comuns = completos["med"][na_min & na_max]
print("Meses completos nas três:", len(datas_comuns))

# Rótulos das linhas de cada base que são meses comuns (o que a avaliacao_mesma_prova recebe)
meses_sorteio = {}
for estatistica in estatisticas:
    base = bases[estatistica]
    meses_sorteio[estatistica] = base.index[base["Data"].isin(datas_comuns)]

# As 3 bases têm os mesmos meses, na mesma ordem, e o mesmo índice (0 a 190, do Excel):
# com as mesmas seeds, as células sorteadas são as mesmas nas 3 (conferência na avaliação)
print("Mesmos rótulos de linha nas 3:",
      meses_sorteio["min"].equals(meses_sorteio["med"]) and meses_sorteio["max"].equals(meses_sorteio["med"]))

Meses completos, min: 118
Meses completos, med: 118
Meses completos, max: 118
Meses completos nas três: 112
Mesmos rótulos de linha nas 3: True


# Conferência da escolha do k
A avaliação usa o k e os pesos escolhidos na média. Aqui, a mesma grade (k = 1, 3 … 15 × pesos; seeds 100 a 129, outras células que não as da avaliação; sorteio nos meses comuns) roda no mínimo e no máximo, só para ver se a escolha seria a mesma. Ela não muda a avaliação.

In [4]:
# Escolha feita na média: a mesma conta do notebook dela (maior R² médio na média das 4 taxas)
escolha_k_media = pd.read_excel(caminho_tratada, sheet_name="escolha_k")
tabela_escolha_media = escolha_k_media.pivot(index=["k", "pesos"], columns="pct_faltantes", values="r2_medio")

# idxmax devolve o rótulo da linha com o maior valor: aqui, o par (k, pesos)
k_escolhido, pesos_escolhidos = tabela_escolha_media.mean(axis=1).idxmax()
print("Escolhido na média (usado na avaliação): k =", k_escolhido, "| pesos =", pesos_escolhidos)

Escolhido na média (usado na avaliação): k = 13 | pesos = distance


In [5]:
grade_k = [1, 3, 5, 7, 9, 11, 13, 15]
grade_pesos = ["uniform", "distance"]
linhas = []

for estatistica in ["min", "max"]:
    for k in grade_k:
        for pesos in grade_pesos:
            # Só o KNN, nas seeds 100 a 129, com a mesma prova. O _ recebe a tabela por rodada, que aqui não é usada
            _, celulas_k = avaliacao_mesma_prova(bases[estatistica], seeds=range(100, 130), metodos=("knn",),
                                                 n_neighbors=k, weights=pesos, devolver_celulas=True,
                                                 meses_sorteio=meses_sorteio[estatistica])
            juntando_k = metricas_juntando(celulas_k)

            # R² médio entre as variáveis, em cada taxa (cada grupo = as 16 variáveis de uma taxa)
            for taxa, grupo in juntando_k.groupby("pct_faltantes"):
                linhas.append({
                    "estatistica": estatistica,
                    "k": k,
                    "pesos": pesos,
                    "pct_faltantes": taxa,
                    "r2_medio": grupo["r2"].mean()
                })

escolha_k = pd.DataFrame(linhas)

Uma linha por k e pesos: o R² médio na média das 4 taxas, no mínimo, no máximo e (do Excel, nos meses completos dela) na média.

In [6]:
# pivot_table com aggfunc="mean": a média das 4 taxas para cada (k, pesos) e estatística
tabela_escolha = escolha_k.pivot_table(index=["k", "pesos"], columns="estatistica", values="r2_medio", aggfunc="mean")
tabela_escolha = tabela_escolha[["min", "max"]]

# Coluna da média: o pandas preenche pelo índice (k, pesos), linha a linha
tabela_escolha["med (Excel)"] = tabela_escolha_media.mean(axis=1)

for estatistica in ["min", "max"]:
    k_est, pesos_est = tabela_escolha[estatistica].idxmax()
    print(f"Escolha no {estatistica}: k = {k_est}, pesos = {pesos_est}")

    # O melhor de cada taxa, para ver se a escolha mudaria com a taxa
    da_estatistica = escolha_k[escolha_k["estatistica"] == estatistica]
    for taxa in taxas:
        da_taxa = da_estatistica[da_estatistica["pct_faltantes"] == taxa]
        melhor = da_taxa.loc[da_taxa["r2_medio"].idxmax()]
        print(f"  melhor a {taxa:.0%}: k = {melhor['k']}, pesos = {melhor['pesos']}")

tabela_escolha.round(3)

Escolha no min: k = 11, pesos = distance
  melhor a 5%: k = 11, pesos = distance
  melhor a 10%: k = 11, pesos = distance
  melhor a 20%: k = 11, pesos = distance
  melhor a 30%: k = 15, pesos = distance
Escolha no max: k = 15, pesos = distance
  melhor a 5%: k = 13, pesos = distance
  melhor a 10%: k = 15, pesos = distance
  melhor a 20%: k = 11, pesos = distance
  melhor a 30%: k = 15, pesos = distance


estatistica    min    max  med (Excel)
k  pesos                              
1  distance -0.306 -0.498       -0.152
   uniform  -0.306 -0.498       -0.152
3  distance  0.107  0.043        0.218
   uniform   0.095  0.035        0.208
5  distance  0.188  0.155        0.271
   uniform   0.173  0.144        0.256
7  distance  0.220  0.194        0.295
   uniform   0.202  0.178        0.278
9  distance  0.233  0.210        0.301
   uniform   0.213  0.191        0.281
11 distance  0.236  0.214        0.305
   uniform   0.214  0.192        0.283
13 distance  0.234  0.218        0.307
   uniform   0.210  0.194        0.284
15 distance  0.232  0.218        0.305
   uniform   0.207  0.193        0.280

# Avaliação
30 rodadas (seeds 0 a 29) × 4 taxas em cada estatística, com os 3 métodos nas mesmas células, o KNN com o k e os pesos da média e o sorteio nos meses comuns.

In [7]:
# Uma tabela de rodadas e uma de células por estatística; concat as empilha numa só,
# com a coluna "estatistica" para separar
rodadas = []
todas_celulas = []
for estatistica in estatisticas:
    tabela_est, celulas_est = avaliacao_mesma_prova(bases[estatistica], n_neighbors=int(k_escolhido),
                                                    weights=pesos_escolhidos, devolver_celulas=True,
                                                    meses_sorteio=meses_sorteio[estatistica])
    tabela_est["estatistica"] = estatistica
    celulas_est["estatistica"] = estatistica
    rodadas.append(tabela_est)
    todas_celulas.append(celulas_est)

tabela = pd.concat(rodadas, ignore_index=True)
celulas = pd.concat(todas_celulas, ignore_index=True)
tabela.head()

,variavel,rmse,mae,r2,n_celulas,metodo,pct_faltantes,seed,estatistica
0,COR,48.261170,37.994690,-0.018052,5,media,0.05,0,min
1,TURB.,6.207986,5.447788,-1.316051,5,media,0.05,0,min
2,pH,0.128974,0.098584,-1.079274,5,media,0.05,0,min
3,ALC.,6.066568,5.414159,-0.062449,5,media,0.05,0,min
4,AC.,3.297137,2.810619,-0.598693,5,media,0.05,0,min


Conferência: número de rodadas, células fora dos meses comuns (precisa ser 0), as mesmas células sorteadas nas 3 estatísticas e células avaliadas por variável em cada taxa.

In [8]:
print("Rodadas:", tabela["seed"].nunique())

chaves = ["seed", "pct_faltantes", "metodo", "variavel", "indice"]
celulas_med = celulas[celulas["estatistica"] == "med"][chaves].reset_index(drop=True)
for estatistica in estatisticas:
    da_estatistica = celulas[celulas["estatistica"] == estatistica]
    fora = (~da_estatistica["indice"].isin(meses_sorteio[estatistica])).sum()

    # As mesmas células da média: a mesma seed, taxa, método, variável e linha, na mesma ordem
    mesmas = da_estatistica[chaves].reset_index(drop=True).equals(celulas_med)
    print(f"{estatistica}: células fora dos meses comuns {fora} | as mesmas células da média: {mesmas}")

# Células por variável em cada rodada (o número é o mesmo em todas as rodadas, métodos e estatísticas)
tabela.groupby(["pct_faltantes", "metodo"], sort=False)["n_celulas"].first().unstack()

Rodadas: 30
min: células fora dos meses comuns 0 | as mesmas células da média: True
med: células fora dos meses comuns 0 | as mesmas células da média: True
max: células fora dos meses comuns 0 | as mesmas células da média: True


metodo,media,mediana,knn
pct_faltantes,,,
0.05,5,5,5
0.10,11,11,11
0.20,22,22,22
0.30,33,33,33


# Métricas juntando as rodadas
Como na média: RMSE, MAE e R² calculados uma vez sobre todas as células de cada método × taxa × variável (`metricas_juntando`, uma vez por estatística), e o DP do RMSE e do MAE entre as 30 rodadas.

In [9]:
resumos = []
for estatistica in estatisticas:
    juntando = metricas_juntando(celulas[celulas["estatistica"] == estatistica])

    # DP entre as rodadas: o groupby separa as rodadas por método, taxa e variável; o std calcula o DP das 30
    rodadas_est = tabela[tabela["estatistica"] == estatistica]
    desvio = rodadas_est.groupby(["metodo", "pct_faltantes", "variavel"], sort=False)[["rmse", "mae"]].std()

    # add_suffix renomeia as colunas (rmse -> rmse_dp); reset_index devolve as chaves do groupby às colunas
    desvio = desvio.add_suffix("_dp").reset_index()

    # merge junta as duas tabelas pelas linhas com o mesmo método, a mesma taxa e a mesma variável
    resumo_est = juntando.merge(desvio, on=["metodo", "pct_faltantes", "variavel"])
    resumo_est.insert(0, "estatistica", estatistica)
    resumos.append(resumo_est)

resumo = pd.concat(resumos, ignore_index=True)
resumo = resumo[["estatistica", "metodo", "pct_faltantes", "variavel", "rmse", "rmse_dp", "mae", "mae_dp",
                 "r2", "n_celulas", "n_seeds"]]
print("Linhas:", len(resumo), "(3 estatísticas × 3 métodos × 4 taxas × 16 variáveis)")
resumo.head()

Linhas: 576 (3 estatísticas × 3 métodos × 4 taxas × 16 variáveis)


,estatistica,metodo,pct_faltantes,variavel,rmse,rmse_dp,mae,mae_dp,r2,n_celulas,n_seeds
0,min,media,0.05,COR,38.611780,18.104358,25.896519,10.820834,-0.016660,150,30
1,min,media,0.05,TURB.,22.627276,11.674188,14.254395,7.176569,-0.032619,150,30
2,min,media,0.05,pH,0.212712,0.077686,0.158643,0.062335,-0.016412,150,30
3,min,media,0.05,ALC.,16.244438,6.882643,12.288968,5.404934,-0.014694,150,30
4,min,media,0.05,AC.,2.531317,0.829522,1.899115,0.740544,-0.019584,150,30


## Visão geral
R² médio entre as 16 variáveis, por estatística, método e taxa, e em quantas das 16 variáveis o KNN teve RMSE menor que a média e que a mediana (nas mesmas células).

In [10]:
# pivot_table com aggfunc="mean": a média do R² das 16 variáveis para cada (estatística, método) e taxa
visao_geral = resumo.pivot_table(index=["estatistica", "metodo"], columns="pct_faltantes", values="r2", aggfunc="mean")

# Estatísticas na ordem mín, méd, máx e métodos na ordem média, mediana, KNN
ordem = []
for estatistica in estatisticas:
    for metodo in metodos:
        ordem.append((estatistica, metodo))
visao_geral = visao_geral.loc[ordem]
print("R² médio entre as variáveis")
display(visao_geral.round(3))

linhas = []
for estatistica in estatisticas:
    for taxa in taxas:
        # Linhas desta estatística e taxa, com (método, variável) como índice, para comparar variável a variável
        parte = resumo[(resumo["estatistica"] == estatistica) & (resumo["pct_faltantes"] == taxa)]
        parte = parte.set_index(["metodo", "variavel"])
        rmse_knn = parte.loc["knn", "rmse"]
        linhas.append({
            "estatistica": estatistica,
            "pct_faltantes": taxa,
            "knn_menor_que_media": (rmse_knn < parte.loc["media", "rmse"]).sum(),
            "knn_menor_que_mediana": (rmse_knn < parte.loc["mediana", "rmse"]).sum()
        })

knn_menor = pd.DataFrame(linhas)
print("Variáveis (de 16) em que o KNN teve RMSE menor")
display(knn_menor)

R² médio entre as variáveis


pct_faltantes         0.05   0.10   0.20   0.30
estatistica metodo                             
min         media   -0.024 -0.022 -0.024 -0.024
            mediana -0.079 -0.067 -0.064 -0.067
            knn      0.225  0.234  0.220  0.183
med         media   -0.027 -0.023 -0.025 -0.025
            mediana -0.123 -0.100 -0.097 -0.099
            knn      0.359  0.355  0.322  0.279
max         media   -0.029 -0.021 -0.023 -0.023
            mediana -0.147 -0.121 -0.117 -0.113
            knn      0.268  0.250  0.215  0.180

Variáveis (de 16) em que o KNN teve RMSE menor


,estatistica,pct_faltantes,knn_menor_que_media,knn_menor_que_mediana
0,min,0.05,13,14
1,min,0.10,13,14
2,min,0.20,14,15
3,min,0.30,14,15
4,med,0.05,16,16
5,med,0.10,16,16
6,med,0.20,16,16
7,med,0.30,15,16
8,max,0.05,14,16
9,max,0.10,14,16


## Por variável (mínimo e máximo)
Para cada estatística, uma tabela por métrica: variáveis nas linhas; uma coluna para cada método e taxa.

In [11]:
for estatistica in ["min", "max"]:
    da_estatistica = resumo[resumo["estatistica"] == estatistica]
    for metrica in ["rmse", "mae", "r2"]:
        # pivot: variáveis nas linhas; colunas em dois níveis (método, taxa)
        tabela_metrica = da_estatistica.pivot(index="variavel", columns=["metodo", "pct_faltantes"], values=metrica)

        # Variáveis na ordem da base e métodos na ordem média, mediana, KNN
        tabela_metrica = tabela_metrica.loc[variaveis, metodos]

        print(f"{estatistica}, {metrica.upper()}")
        display(tabela_metrica.round(3))

min, RMSE


metodo             media                                     mediana  \
pct_faltantes       0.05       0.10       0.20       0.30       0.05   
variavel                                                               
COR               38.612     36.629     35.225     35.720     39.268   
TURB.             22.627     19.523     19.049     19.736     24.110   
pH                 0.213      0.223      0.220      0.219      0.212   
ALC.              16.244     16.286     16.424     16.470     16.754   
AC.                2.531      2.592      2.631      2.583      2.523   
O.C.               1.532      1.562      1.543      1.535      1.542   
O.D.               0.814      0.788      0.814      0.820      0.811   
Cl                11.494     11.870     11.894     11.761     11.490   
DUR.              13.408     12.760     12.844     12.588     13.069   
Fe                 1.020      1.094      1.071      1.071      1.059   
Mn                 0.043      0.041      0.041      0.041      0.043   
Cond.            105.843    103.079     99.634    101.131    111.431   
Cianobacteria  11049.093  10048.858   9919.207  10021.326  12246.992   
C.F.           61196.899  49745.224  49460.072  46378.786  64480.456   
Clorofila         29.311     27.634     27.517     27.723     31.059   
F                  0.080      0.084      0.088      0.087      0.081   

metodo                                                knn             \
pct_faltantes       0.10       0.20       0.30       0.05       0.10   
variavel                                                               
COR               37.390     35.454     36.109     25.337     25.772   
TURB.             20.452     19.801     20.689     16.085     14.492   
pH                 0.222      0.218      0.217      0.195      0.200   
ALC.              16.588     16.791     16.842     12.197     11.794   
AC.                2.593      2.631      2.594      2.691      2.708   
O.C.               1.565      1.573      1.566      1.555      1.570   
O.D.               0.792      0.815      0.822      0.685      0.651   
Cl                11.884     11.886     11.709      9.308      9.519   
DUR.              12.678     12.764     12.483     11.658     10.818   
Fe                 1.142      1.118      1.117      0.755      0.809   
Mn                 0.042      0.041      0.042      0.043      0.041   
Cond.            106.371    101.382    103.131     68.266     68.480   
Cianobacteria  10712.921  10451.280  10586.016  11149.240   9666.836   
C.F.           52529.227  51572.610  48564.079  59443.205  48330.469   
Clorofila         28.866     29.144     29.522     26.136     25.221   
F                  0.084      0.088      0.087      0.071      0.073   

metodo                               
pct_faltantes       0.20       0.30  
variavel                             
COR               25.230     27.413  
TURB.             14.161     15.372  
pH                 0.204      0.209  
ALC.              12.378     13.036  
AC.                2.712      2.701  
O.C.               1.522      1.544  
O.D.               0.688      0.701  
Cl                 9.443      9.642  
DUR.              10.758     10.530  
Fe                 0.816      0.856  
Mn                 0.041      0.041  
Cond.             68.643     75.940  
Cianobacteria   9539.280   9785.307  
C.F.           48596.388  45686.931  
Clorofila         26.120     26.544  
F                  0.077      0.078

min, MAE


metodo             media                                     mediana  \
pct_faltantes       0.05       0.10       0.20       0.30       0.05   
variavel                                                               
COR               25.897     25.229     24.749     24.991     24.333   
TURB.             14.254     12.307     12.135     12.570     12.880   
pH                 0.159      0.170      0.166      0.165      0.158   
ALC.              12.289     12.552     12.727     12.671     12.513   
AC.                1.899      1.975      2.017      1.968      1.887   
O.C.               1.269      1.280      1.273      1.270      1.245   
O.D.               0.618      0.635      0.656      0.656      0.616   
Cl                 8.828      9.053      9.226      9.130      8.867   
DUR.              11.097     10.307     10.460     10.245     10.767   
Fe                 0.685      0.730      0.704      0.699      0.598   
Mn                 0.035      0.033      0.032      0.033      0.035   
Cond.             83.523     82.464     80.386     80.585     85.613   
Cianobacteria   8516.352   7771.451   7719.936   7761.086   7584.033   
C.F.           29597.037  25230.871  24369.610  23424.184  26371.800   
Clorofila         22.210     21.147     20.981     21.098     20.937   
F                  0.063      0.066      0.069      0.068      0.064   

metodo                                                knn             \
pct_faltantes       0.10       0.20       0.30       0.05       0.10   
variavel                                                               
COR               23.761     22.905     23.173     17.941     17.946   
TURB.             10.503     10.073     10.693      8.455      7.477   
pH                 0.170      0.165      0.163      0.141      0.153   
ALC.              12.591     12.741     12.679      8.723      8.756   
AC.                1.979      2.023      1.975      2.055      2.061   
O.C.               1.254      1.266      1.261      1.249      1.262   
O.D.               0.637      0.656      0.655      0.509      0.505   
Cl                 9.097      9.227      9.067      6.945      7.161   
DUR.              10.176     10.329     10.128      8.458      7.894   
Fe                 0.654      0.628      0.619      0.431      0.481   
Mn                 0.033      0.032      0.033      0.032      0.031   
Cond.             82.852     79.212     79.402     51.276     51.536   
Cianobacteria   6351.191   6195.176   6176.431   7283.275   6088.772   
C.F.           21706.424  19268.303  18450.737  26472.079  22071.008   
Clorofila         19.822     20.054     20.246     17.918     17.239   
F                  0.066      0.070      0.068      0.055      0.056   

metodo                               
pct_faltantes       0.20       0.30  
variavel                             
COR               17.645     18.332  
TURB.              7.463      8.211  
pH                 0.155      0.160  
ALC.               9.250      9.688  
AC.                2.076      2.066  
O.C.               1.231      1.269  
O.D.               0.538      0.548  
Cl                 7.048      7.280  
DUR.               8.045      8.119  
Fe                 0.472      0.500  
Mn                 0.031      0.032  
Cond.             52.007     55.875  
Cianobacteria   6184.105   6299.241  
C.F.           21603.336  21073.560  
Clorofila         18.546     18.629  
F                  0.060      0.060

min, R2


metodo         media                      mediana                         knn  \
pct_faltantes   0.05   0.10   0.20   0.30    0.05   0.10   0.20   0.30   0.05   
variavel                                                                        
COR           -0.017 -0.023 -0.026 -0.029  -0.052 -0.066 -0.039 -0.052  0.562   
TURB.         -0.033 -0.012 -0.023 -0.021  -0.172 -0.111 -0.105 -0.122  0.478   
pH            -0.016 -0.019 -0.018 -0.022  -0.011 -0.011 -0.006 -0.004  0.144   
ALC.          -0.015 -0.019 -0.028 -0.026  -0.079 -0.057 -0.075 -0.073  0.428   
AC.           -0.020 -0.025 -0.016 -0.016  -0.013 -0.027 -0.016 -0.025 -0.152   
O.C.          -0.014 -0.020 -0.031 -0.032  -0.028 -0.024 -0.071 -0.074 -0.045   
O.D.          -0.020 -0.025 -0.021 -0.025  -0.014 -0.033 -0.024 -0.029  0.277   
Cl            -0.017 -0.010 -0.017 -0.023  -0.017 -0.013 -0.016 -0.014  0.333   
DUR.          -0.065 -0.027 -0.022 -0.028  -0.012 -0.013 -0.009 -0.011  0.195   
Fe            -0.012 -0.020 -0.022 -0.018  -0.092 -0.111 -0.114 -0.108  0.445   
Mn            -0.019 -0.023 -0.016 -0.015  -0.032 -0.049 -0.041 -0.027 -0.015   
Cond.         -0.035 -0.016 -0.024 -0.024  -0.147 -0.082 -0.061 -0.064  0.569   
Cianobacteria -0.026 -0.027 -0.044 -0.040  -0.261 -0.167 -0.160 -0.161 -0.045   
C.F.          -0.043 -0.038 -0.026 -0.021  -0.158 -0.157 -0.115 -0.120  0.016   
Clorofila     -0.017 -0.025 -0.026 -0.028  -0.142 -0.118 -0.151 -0.166  0.191   
F             -0.017 -0.019 -0.018 -0.022  -0.031 -0.028 -0.016 -0.021  0.212   

metodo                              
pct_faltantes   0.10   0.20   0.30  
variavel                            
COR            0.493  0.474  0.394  
TURB.          0.442  0.435  0.380  
pH             0.179  0.124  0.074  
ALC.           0.466  0.416  0.357  
AC.           -0.120 -0.080 -0.110  
O.C.          -0.031 -0.004 -0.044  
O.D.           0.302  0.271  0.252  
Cl             0.350  0.359  0.312  
DUR.           0.262  0.283  0.281  
Fe             0.442  0.406  0.350  
Mn            -0.034 -0.020  0.001  
Cond.          0.551  0.514  0.423  
Cianobacteria  0.049  0.034  0.008  
C.F.           0.020  0.010  0.009  
Clorofila      0.147  0.076  0.057  
F              0.219  0.216  0.180

... (2 outputs left out)

max, MAE


metodo             media                                     mediana  \
pct_faltantes       0.05       0.10       0.20       0.30       0.05   
variavel                                                               
COR              199.720    193.226    189.226    191.445    193.073   
TURB.            218.947    227.144    224.016    224.913    196.540   
pH                 0.146      0.148      0.145      0.145      0.141   
ALC.              22.362     21.249     21.625     22.055     21.853   
AC.                6.375      6.422      6.293      6.517      5.660   
O.C.               3.543      3.626      3.695      3.657      3.269   
O.D.               0.896      0.967      1.003      1.011      0.887   
Cl                15.015     16.265     15.747     15.258     13.960   
DUR.              18.249     18.831     19.746     19.362     16.467   
Fe                 2.634      2.808      2.878      2.995      2.604   
Mn                 0.155      0.163      0.162      0.164      0.137   
Cond.            136.887    128.306    127.690    128.785    144.413   
Cianobacteria   9288.812   8614.313   8411.376   8554.745   9163.020   
C.F.           42247.856  38042.762  37639.020  35796.595  37283.133   
Clorofila         20.695     19.480     19.531     19.715     19.756   
F                  0.155      0.174      0.164      0.168      0.151   

metodo                                                knn             \
pct_faltantes       0.10       0.20       0.30       0.05       0.10   
variavel                                                               
COR              181.021    175.817    174.714    105.600     99.721   
TURB.            189.697    187.935    192.289    117.195    124.666   
pH                 0.143      0.139      0.139      0.119      0.126   
ALC.              20.485     20.983     21.341     13.637     12.766   
AC.                5.500      5.382      5.519      5.860      5.731   
O.C.               3.403      3.470      3.439      3.007      3.114   
O.D.               0.953      0.985      0.997      0.714      0.778   
Cl                15.300     14.706     14.374     10.694     12.101   
DUR.              17.961     18.797     18.446     14.959     16.655   
Fe                 2.778      2.740      2.824      1.391      1.630   
Mn                 0.144      0.141      0.142      0.117      0.128   
Cond.            131.761    129.818    130.022     88.731     82.779   
Cianobacteria   7560.976   7295.286   7305.594   8672.642   8027.826   
C.F.           31750.667  29464.818  27806.697  45201.852  40404.095   
Clorofila         18.541     18.731     19.121     16.470     15.674   
F                  0.170      0.160      0.165      0.148      0.165   

metodo                               
pct_faltantes       0.20       0.30  
variavel                             
COR              102.184    112.035  
TURB.            133.233    145.247  
pH                 0.129      0.130  
ALC.              13.255     14.784  
AC.                5.776      6.193  
O.C.               3.174      3.171  
O.D.               0.857      0.884  
Cl                11.995     12.224  
DUR.              17.647     17.529  
Fe                 1.737      1.950  
Mn                 0.129      0.137  
Cond.             78.533     84.790  
Cianobacteria   7793.545   7935.681  
C.F.           38597.830  36067.335  
Clorofila         16.829     17.145  
F                  0.152      0.157

max, R2


metodo         media                      mediana                         knn  \
pct_faltantes   0.05   0.10   0.20   0.30    0.05   0.10   0.20   0.30   0.05   
variavel                                                                        
COR           -0.021 -0.030 -0.035 -0.052  -0.188 -0.169 -0.178 -0.156  0.569   
TURB.         -0.020 -0.013 -0.016 -0.019  -0.339 -0.252 -0.242 -0.253  0.502   
pH            -0.014 -0.008 -0.011 -0.018  -0.110 -0.098 -0.092 -0.090  0.280   
ALC.          -0.028 -0.015 -0.024 -0.022  -0.123 -0.067 -0.084 -0.077  0.566   
AC.           -0.018 -0.018 -0.025 -0.019  -0.140 -0.115 -0.125 -0.098  0.042   
O.C.          -0.040 -0.024 -0.027 -0.030  -0.052 -0.067 -0.078 -0.086  0.116   
O.D.          -0.029 -0.036 -0.023 -0.027  -0.029 -0.022 -0.014 -0.019  0.256   
Cl            -0.012 -0.017 -0.014 -0.018  -0.070 -0.074 -0.062 -0.078  0.310   
DUR.          -0.066 -0.032 -0.015 -0.018  -0.034 -0.081 -0.080 -0.072  0.163   
Fe            -0.027 -0.024 -0.026 -0.016  -0.224 -0.225 -0.196 -0.185  0.588   
Mn            -0.017 -0.016 -0.017 -0.018  -0.246 -0.230 -0.217 -0.206  0.194   
Cond.         -0.051 -0.017 -0.023 -0.017  -0.157 -0.079 -0.065 -0.063  0.424   
Cianobacteria -0.030 -0.022 -0.042 -0.036  -0.327 -0.181 -0.174 -0.163 -0.033   
C.F.          -0.049 -0.034 -0.026 -0.020  -0.164 -0.140 -0.108 -0.102 -0.060   
Clorofila     -0.019 -0.019 -0.022 -0.024  -0.090 -0.081 -0.104 -0.122  0.235   
F             -0.023 -0.015 -0.019 -0.017  -0.053 -0.050 -0.047 -0.044  0.131   

metodo                              
pct_faltantes   0.10   0.20   0.30  
variavel                            
COR            0.549  0.487  0.419  
TURB.          0.482  0.418  0.346  
pH             0.228  0.150  0.128  
ALC.           0.559  0.533  0.455  
AC.            0.054  0.017 -0.021  
O.C.           0.080  0.059  0.038  
O.D.           0.228  0.148  0.138  
Cl             0.292  0.289  0.230  
DUR.           0.079  0.042  0.040  
Fe             0.557  0.509  0.423  
Mn             0.226  0.170  0.122  
Cond.          0.474  0.520  0.470  
Cianobacteria -0.055 -0.087 -0.083  
C.F.          -0.075 -0.063 -0.034  
Clorofila      0.210  0.144  0.126  
F              0.115  0.105  0.086

# Gravação
Abas `escolha_k` (a grade no mínimo e no máximo), `resumo` (métricas juntando + DP do RMSE e do MAE, com a coluna `estatistica`), `rodadas` (uma linha por estatística × seed × taxa × método × variável), `visao_geral` (o R² médio) e `knn_menor`. Rodar de novo dá os mesmos números: as seeds são fixas, e o KNN não tem sorteio.

In [12]:
caminho = BASE_DIR / "data" / "output" / "imputacoes_mesma_prova_min_max.xlsx"

# ExcelWriter permite gravar várias abas no mesmo arquivo; na visao_geral, reset_index devolve
# a estatística e o método às colunas
with pd.ExcelWriter(caminho) as arquivo:
    escolha_k.to_excel(arquivo, sheet_name="escolha_k", index=False)
    resumo.to_excel(arquivo, sheet_name="resumo", index=False)
    tabela.to_excel(arquivo, sheet_name="rodadas", index=False)
    visao_geral.reset_index().to_excel(arquivo, sheet_name="visao_geral", index=False)
    knn_menor.to_excel(arquivo, sheet_name="knn_menor", index=False)

print("Gravado em:", caminho)

Gravado em: C:\_Master\Programacao\Guilherme-Betta\1_Projects\IC2\IC2-item-7\data\output\imputacoes_mesma_prova_min_max.xlsx
